In [ ]:
%load_ext autoreload
%autoreload 3 -p

In [ ]:
import colorcet as cc
import holoviews as hv
import numpy as np
import pandas as pd
import xarray
from holoviews import opts
from joblib import delayed
from omegaconf import OmegaConf
from tqdm_joblib import ParallelPbar

import gnss_tid.image
import gnss_tid.pointdata
import gnss_tid.spectral

hv.extension("bokeh")
opts.defaults(
    opts.Image(data_aspect=1, frame_height=400, frame_width=600, cmap=cc.cm.coolwarm, colorbar=True, show_grid=True, tools=["hover"]),
    hv.opts.Scatter(data_aspect=1, frame_height=400, frame_width=600, cmap=cc.cm.coolwarm, colorbar=True, show_grid=True, tools=["hover"]),
)

In [ ]:
# cfg = OmegaConf.load("apps/conf/event/20170820.yaml")
# cfg = OmegaConf.load("apps/conf/event/20160508.yaml")
cfg = OmegaConf.load("apps/conf/event/20210101.yaml")
# cfg = OmegaConf.load("apps/conf/event/20150325_full.yaml")
cfg

In [ ]:
lat_limits=cfg.latitude_limits
lon_limits=cfg.longitude_limits
time_limits=cfg.time_limits

obs, rx = gnss_tid.pointdata.load_observations(
    files=cfg.files,
    lat_limits=lat_limits,
    lon_limits=lon_limits,
    time_limits=time_limits,
    el_min=30,
)

In [ ]:
# med_stec = obs["stec"].abs().groupby(obs["rx"]).median()
# good = med_stec <= 30
# obs = obs[obs["rx"].map(good)]

In [ ]:
proj = gnss_tid.pointdata.Local2D.from_geodetic(np.mean(lat_limits), np.mean(lon_limits), 300.0)
windows = gnss_tid.pointdata.make_time_windows(obs, 10, 30)
maker = gnss_tid.image.ScipyRbfImageMaker(
    hres=20,                # grid resolution (km / pixel)
    hp_freq=0.03,           # Butterworth high-pass cutoff (cycles / pixel)
    neighbor_radius=100.0,  # density-mask radius (km)
    neighbors=50,           # RBF neighborhood size
    kernel="multiquadric",
    smoothing=0.2,
)
maker.initialize_from_bounds(proj, lat_limits, lon_limits)

In [ ]:
PLOT_TYPE = "time"
N_JOBS = 8
HEIGHT = 350.0
heights = np.arange(150, 600, 10)
twin = windows[16]

def ft(t):
    return gnss_tid.pointdata.get_data(obs, rx, t, HEIGHT, lat_limits, lon_limits, proj)
def fh(h):
    return gnss_tid.pointdata.get_data(obs, rx, twin, h, lat_limits, lon_limits, proj)


if PLOT_TYPE == "time":
    dfs = ParallelPbar("making plots")(n_jobs=N_JOBS)(
        delayed(ft)(t) for t in windows
    )
    slider = [t.start_time for t in windows]
    title = f"height = {HEIGHT}km"
else:
    dfs = ParallelPbar("making plots")(n_jobs=N_JOBS)(
        delayed(fh)(h) for h in heights
    )
    slider = heights
    title = np.datetime_as_string(twin.start_time, unit="m")


In [ ]:
CLIM = .5

plots = {
    x: hv.Scatter(df[df["dtec3"].notna()], kdims="x", vdims=["y", "dtec3"]).opts(size=3.5, color="dtec3", clim=(-CLIM, CLIM), title=f"{title} | {str(x)}")
    for x, df in zip(slider, dfs, strict=True)
}
hv.output(widget_location='bottom')
hv.HoloMap(plots, kdims=[PLOT_TYPE])

In [ ]:
# Downstream NaN diagnosis. A red receiver has >=10% NaN arc points *after*
# the time-window mean, so this directly measures the points that disappear
# from the image/scatter pipeline rather than raw NaNs in the source files.
def arc_nan_stats(windows):
    """Compute downstream dtec1 NaN fractions after window aggregation.

    Args:
        windows: TimeWindow list from gnss_tid.pointdata.make_time_windows.

    Returns:
        Tuple of four values: the fraction of all-NaN arc points per receiver,
        the same fraction per satellite, the raw dtec1 sample-NaN fraction per
        receiver, and the per-(rx, sv) dead-arc boolean Series. An arc point is
        NaN only when every sample of an ``(rx, sv)`` group in a window is NaN.
    """
    arc_counts = []
    for window in windows:
        sample = obs.iloc[window.row_indices]
        arc_counts.append(sample.groupby(["rx", "sv"])["dtec1"].count())
    finite_per_arc = pd.concat(arc_counts)
    dead_arc = finite_per_arc.eq(0)
    return (
        dead_arc.groupby(level="rx").mean(),
        dead_arc.groupby(level="sv").mean(),
        obs["dtec1"].isna().groupby(obs["rx"]).mean(),
        dead_arc,
    )


receiver_arc_nan, satellite_arc_nan, receiver_sample_nan, dead_arc = arc_nan_stats(windows)
receiver_threshold = 0.10
satellite_threshold = 0.05

receiver_locations = rx.drop_duplicates("rx").set_index("rx")
in_aoi = receiver_locations["lat"].between(lat_limits[0], lat_limits[1]) & receiver_locations["lon"].between(lon_limits[0], lon_limits[1])
receiver_stats = (
    receiver_locations.loc[in_aoi]
    .join(receiver_arc_nan.rename("arc_nan_fraction"))
    .join(receiver_sample_nan.rename("raw_sample_nan_fraction"))
    .dropna(subset=["arc_nan_fraction"])
)
healthy_receivers = receiver_stats.query("arc_nan_fraction < @receiver_threshold")
affected_receivers = receiver_stats.query("arc_nan_fraction >= @receiver_threshold")

print(
    f"{len(affected_receivers):,} of {len(receiver_stats):,} AOI receivers lose "
    f">= {receiver_threshold:.0%} of their downstream arc points to NaN."
)
print(
    f"Worst satellite: {satellite_arc_nan.idxmax()} loses "
    f"{satellite_arc_nan.max():.1%} of downstream arc points."
)

receiver_background = hv.Points(
    healthy_receivers.reset_index(), kdims=["lon", "lat"], label="<10% NaN arc points"
).opts(color="#d9d9d9", size=4, alpha=0.55)
receiver_alerts = hv.Points(
    affected_receivers.reset_index(),
    kdims=["lon", "lat"],
    vdims=["arc_nan_fraction", "raw_sample_nan_fraction"],
    label=">=10% NaN arc points",
).opts(
    color="arc_nan_fraction",
    cmap=cc.cm.linear_worb_100_25_c53,
    clim=(receiver_threshold, 1),
    size=6,
    colorbar=True,
    alpha=0.95,
    tools=["hover"],
)
receiver_map = (receiver_background * receiver_alerts).opts(
    frame_width=500,
    frame_height=320,
    xlabel="longitude (deg)",
    ylabel="latitude (deg)",
    title="Downstream NaN arc points cluster in a small set of AOI receivers",
    legend_position="top_right",
)

satellite_stats = satellite_arc_nan.rename("arc_nan_fraction").sort_values(ascending=False)
satellites_ok = satellite_stats[satellite_stats < satellite_threshold].reset_index()
satellites_bad = satellite_stats[satellite_stats >= satellite_threshold].reset_index()
satellite_bars = (
    hv.Bars(satellites_ok, kdims=["sv"], vdims="arc_nan_fraction", label="<5% NaN")
    .opts(color="#bdbdbd", alpha=0.8)
    * hv.Bars(satellites_bad, kdims=["sv"], vdims="arc_nan_fraction", label=">=5% NaN")
    .opts(color="#d62728", alpha=0.95)
    * hv.HLine(satellite_threshold).opts(color="black", line_dash="dashed", line_width=2)
).opts(
    frame_width=400,
    frame_height=305,
    xrotation=45,
    xlabel="satellite",
    ylabel="downstream NaN arc-point fraction",
    title="A few satellites dominate the downstream NaN loss (dashed = 5%)",
    legend_position="top_left",
    tools=["hover"],
)

hv.Layout(receiver_map + satellite_bars).opts(shared_axes=False)

In [ ]:
# Large |dtec1| values are concentrated in a few receivers with STEC values
# thousands of times larger than a clean receiver. The panels below use
# explicit thresholds and direct good-vs-bad scale comparisons.
mag_rx = obs["dtec1"].abs().groupby(obs["rx"]).median()
med_stec = obs["stec"].abs().groupby(obs["rx"]).median()
n_rx = obs.groupby("rx").size()
finite_stec = obs.groupby("rx")["stec"].count()
plenty = n_rx >= 1000
physical_stec = med_stec.reindex(n_rx.index) < 100
garbage_stec = med_stec.reindex(n_rx.index) > 100
mostly_finite_stec = finite_stec.reindex(n_rx.index) > 0.5 * n_rx
good_rx = mag_rx.loc[n_rx[plenty & physical_stec].index].idxmin()
bad_stec = med_stec.loc[n_rx[plenty & garbage_stec & mostly_finite_stec].index].nlargest(5)
print(f"Clean reference {good_rx}: median |STEC|={med_stec[good_rx]:.1f}, median |dtec1|={mag_rx[good_rx]:.3f}.")
print("Five garbage-scale receivers (median |STEC|):")
print(bad_stec.to_string(float_format=lambda value: f"{value:,.0f}"))

def stec_arc(name):
    """Return the arc with the most finite STEC samples for one receiver.

    Args:
        name: Receiver identifier in the ``rx`` column of ``obs``.

    Returns:
        DataFrame with ``time``, ``stec`` and ``dtec1`` for the satellite arc
        having the most finite STEC samples, or None when none are finite.
    """
    receiver_obs = obs[obs["rx"] == name]
    finite_counts = receiver_obs[receiver_obs["stec"].notna()].groupby("sv").size()
    if finite_counts.empty:
        return None
    return receiver_obs[receiver_obs["sv"] == finite_counts.idxmax()].sort_values("time").loc[:, ["time", "stec", "dtec1"]]

bad_arcs = {name: stec_arc(name) for name in bad_stec.index}
good_arc = stec_arc(good_rx)
outlier_threshold = 1.0
window = windows[len(windows) // 2]
window_points = gnss_tid.pointdata.get_data(obs, rx, window, 350.0, lat_limits, lon_limits, proj)
finite_points = window_points[window_points["dtec1"].notna()]
normal_points = finite_points[finite_points["dtec1"].abs() <= outlier_threshold]
outlier_points = finite_points[finite_points["dtec1"].abs() > outlier_threshold]
print(f"{len(outlier_points):,}/{len(finite_points):,} arc points in {np.datetime_as_string(window.start_time, unit='m')} exceed |dtec1|=1 TECU.")

normal_map = hv.Points(normal_points, kdims=["x", "y"], label="|dtec1| <= 1 TECU").opts(color="#d9d9d9", size=3, alpha=0.25)
red_outliers = hv.Points(outlier_points, kdims=["x", "y"], vdims=["rx", "sv", "dtec1", "stec"], label="|dtec1| > 1 TECU").opts(color="#d62728", size=6, alpha=0.9, tools=["hover"])
outlier_map = (normal_map * red_outliers).opts(frame_width=700, frame_height=420, data_aspect=1, xlabel="x (km)", ylabel="y (km)", title="Spatial clusters of impossible-scale dtec1 values (red = |dtec1| > 1 TECU)", legend_position="top_left")

scale_data = pd.concat([bad_stec.rename("median_abs_stec"), pd.Series({good_rx: med_stec[good_rx]}, name="median_abs_stec")]).sort_values(ascending=False)
scale_frame = scale_data.rename_axis("rx").rename("median_abs_stec").reset_index()
scale_plot = hv.Bars(scale_frame, kdims=["rx"], vdims="median_abs_stec").opts(color="#d62728", frame_width=700, frame_height=300, xrotation=45, xlabel="receiver", ylabel="median |STEC| (file units)", title="Bad receivers are 10^3–10^4 times larger than the clean reference", tools=["hover"])

bad_overlay = hv.Overlay([hv.Curve(bad_arcs[name], "time", "stec", label=name) for name in bad_stec.index if bad_arcs[name] is not None]).opts(frame_width=700, frame_height=320, title="Five bad receiver arcs: garbage-scale STEC", legend_position="right", xlabel="time", ylabel="STEC (file units)", tools=["hover"])
good_curve = hv.Curve(good_arc, "time", "stec", label=good_rx).opts(frame_width=700, frame_height=320, title="Clean receiver arc: physical-scale STEC", xlabel="time", ylabel="STEC (file units)", tools=["hover"])

receiver_dtec = mag_rx.loc[n_rx[plenty].index].nlargest(20)
receiver_frame = receiver_dtec.rename_axis("rx").rename("median_abs_dtec1").reset_index()
receiver_bars = (hv.Bars(receiver_frame, kdims=["rx"], vdims="median_abs_dtec1").opts(color="#d62728", alpha=0.9) * hv.HLine(outlier_threshold).opts(color="black", line_dash="dashed", line_width=2)).opts(frame_width=700, frame_height=300, xrotation=45, xlabel="receiver", ylabel="median |dtec1| (TECU)", title="A small set of receivers dominates the |dtec1| > 1 TECU contamination", tools=["hover"])

hv.Layout(outlier_map + scale_plot + bad_overlay + good_curve + receiver_bars).opts(shared_axes=False).cols(2)

In [ ]:
# Compact readout for the two visual diagnostics above.
diagnostic_readout = pd.DataFrame(
    {
        "problem": [
            "NaN arc-point loss",
            "Worst satellite NaN loss",
            "High-magnitude arc points",
            "Bad-vs-clean STEC scale gap",
        ],
        "evidence": [
            f"{len(affected_receivers):,}/{len(receiver_stats):,} AOI receivers lose >=10% of arc points",
            f"{satellite_arc_nan.idxmax()}: {satellite_arc_nan.max():.1%} arc points lost",
            f"{len(outlier_points):,}/{len(finite_points):,} points exceed |dtec1|=1 TECU",
            f"{bad_stec.iloc[0] / med_stec[good_rx]:,.0f}x median |STEC|",
        ],
    }
)
diagnostic_readout

In [ ]:
# Receiver-level scatter tying the two scale diagnostics together: receivers
# whose median |STEC| exceeds the 100-file-unit physical threshold are also
# the ones whose median |dtec1| crosses the 1 TECU outlier threshold.
# Receivers losing <5% of downstream arc points to NaN are dark grey and
# small; the rest are colored by their NaN arc-point fraction.
stec_scatter_frame = pd.DataFrame(
    {
        "median_abs_stec": med_stec,
        "median_abs_dtec1": mag_rx,
        "n_samples": n_rx,
    }
).dropna()
stec_scatter_frame = stec_scatter_frame.join(receiver_arc_nan.rename("arc_nan_fraction")).dropna()
stec_garbage = stec_scatter_frame["median_abs_stec"] > 100
print(
    f"{stec_garbage.sum():,} of {len(stec_scatter_frame):,} receivers have "
    f"median |STEC| > 100 (garbage scale); "
    f"{(stec_scatter_frame.loc[stec_garbage, 'median_abs_dtec1'] > outlier_threshold).sum():,} "
    f"of them also have median |dtec1| > {outlier_threshold:g} TECU."
)

nan_threshold = 0.1
low_nan_frame = stec_scatter_frame[stec_scatter_frame["arc_nan_fraction"] < nan_threshold]
high_nan_frame = stec_scatter_frame[stec_scatter_frame["arc_nan_fraction"] >= nan_threshold]
print(f"{len(high_nan_frame):,}/{len(stec_scatter_frame):,} receivers lose >= {nan_threshold:.0%} of downstream arc points to NaN.")

low_nan_points = hv.Points(
    low_nan_frame.reset_index(),
    kdims=["median_abs_stec", "median_abs_dtec1"],
    vdims=["n_samples", "arc_nan_fraction"],
    label=f"<{nan_threshold:.0%} NaN arc points",
).opts(color="#535353", size=3, data_aspect=None)
high_nan_points = hv.Points(
    high_nan_frame.reset_index(),
    kdims=["median_abs_stec", "median_abs_dtec1"],
    vdims=["n_samples", "arc_nan_fraction"],
    label=f">={nan_threshold:.0%} NaN arc points",
).opts(
    color="arc_nan_fraction",
    cmap=cc.cm.bmy,
    clim=(nan_threshold, 1),
    colorbar=True,
    size=5,
    alpha=0.95,
    data_aspect=None,
    tools=["hover"],
)

stec_scatter = (low_nan_points * high_nan_points).opts(
    frame_width=500,
    frame_height=400,
    logx=True,
    logy=True,
    xlabel="median |STEC| (file units, log scale)",
    ylabel="median |dtec1| (TECU)",
    legend_position="top_left",
    tools=["hover"],
    show_grid=True,
)
stec_scatter

In [ ]:
# Same downstream-NaN diagnosis as the satellite bars above, but weighted by
# arc counts: a satellite that loses 100% of 2 arcs matters less than one
# that loses 30% of 400. The scatter puts one satellite per point: y is the
# absolute number of downstream NaN arc points it contributes and x is its
# total arc-point count, so the satellites the NaNs actually come from sit
# at the top regardless of their NaN fraction. Satellites with zero NaN arc
# points are excluded from the log-log scatter; the bars rank them directly.
# Reuses the per-arc dead-arc table computed by arc_nan_stats above.
sv_total = dead_arc.groupby(level="sv").size().rename("n_arc_points")
sv_nan = dead_arc.groupby(level="sv").sum().rename("n_nan_arc_points")
sv_stats = pd.concat([sv_total, sv_nan], axis=1)
sv_stats["nan_fraction"] = sv_stats["n_nan_arc_points"] / sv_stats["n_arc_points"]
sv_stats["nan_share"] = sv_stats["n_nan_arc_points"] / sv_stats["n_nan_arc_points"].sum()
sv_stats = sv_stats.sort_values("n_nan_arc_points", ascending=False)

total_nan = int(sv_stats["n_nan_arc_points"].sum())
top5_share = sv_stats["nan_share"].head(5).sum()
print(
    f"{total_nan:,} downstream NaN arc points across {len(sv_stats)} satellites; "
    f"the top 5 satellites contribute {top5_share:.0%} of them."
)
print(sv_stats.head(10).to_string(float_format=lambda value: f"{value:.3f}"))

sv_frame = sv_stats[sv_stats["n_nan_arc_points"] > 0].reset_index()
x_span = np.array([sv_frame["n_arc_points"].min(), sv_frame["n_arc_points"].max()])
fraction_lines = hv.Overlay(
    [
        hv.Curve(pd.DataFrame({"n_arc_points": x_span, "n_nan_arc_points": fraction * x_span}), label=f"{fraction:.0%} NaN").opts(
            color="black", line_dash="dashed", line_width=1, alpha=0.6
        )
        for fraction in (1.0, 0.5, 0.1)
    ]
)

sv_points = hv.Points(
    sv_frame,
    kdims=["n_arc_points", "n_nan_arc_points"],
    vdims=["sv", "nan_fraction", "nan_share"],
    label="satellites",
).opts(
    color="nan_fraction",
    cmap=cc.cm.bmy,
    clim=(0, 1),
    colorbar=True,
    size=6,
    alpha=0.95,
    data_aspect=None,
    logx=True,
    logy=True,
    tools=["hover"],
    show_grid=True,
    xlabel="total arc points (log scale)",
    ylabel="downstream NaN arc points (log scale)",
    title="Which satellites the downstream NaNs actually come from",
    legend_position="top_left",
)
sv_scatter = (fraction_lines * sv_points).opts(frame_width=500, frame_height=400)

top_sv = sv_stats.head(15).reset_index()
sv_bars = hv.Bars(top_sv, kdims=["sv"], vdims=["n_nan_arc_points", "nan_fraction", "nan_share"]).opts(
    color="#d62728",
    alpha=0.9,
    xrotation=45,
    tools=["hover"],
    xlabel="satellite",
    ylabel="downstream NaN arc points",
    title="Top 15 satellites by contributed NaN arc points",
    frame_width=500,
    frame_height=400,
)

hv.Layout(sv_scatter + sv_bars).opts(shared_axes=False)